# Controllable document ingestion

Create asynchronous text/HTML jobs, inspect a review pause, resume, and publish a searchable lexical index. SQLite job records and transactions are owned by this application.

## 1. Open the source

Locally, launch Jupyter from the extracted project. In Google Colab, the next cell asks for the complete project ZIP if its source is not already present. Do not upload only this notebook. This notebook imports the same `tutorials/document_ingestion.py` implementation that the documentation tests execute.

In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## 2. Check optional dependencies

The framework package and its dependencies are unchanged. Colab installs only missing non-PyTorch tutorial packages. Existing PyTorch installations are never replaced by this cell.

In [ ]:
required = {'packaging': 'packaging>=25', 'fastapi': 'fastapi', 'httpx': 'httpx'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")


## 3. Inspect and run

The job moves through parsed, embedded, and published stages. The small deterministic hashing encoder is lexical, not a learned semantic encoder.

In [ ]:
import asyncio
import inspect
from tutorials import document_ingestion as lesson

# Inspect the actual source used by the demonstration and documentation tests.
print(inspect.getsource(lesson.build_graph))

# Jupyter already owns an event loop. Run this synchronous CLI example off that loop.
result = await asyncio.to_thread(lesson.run_demo)
result


## 4. Interpret the result

The job moves through parsed, embedded, and published stages. The small deterministic hashing encoder is lexical, not a learned semantic encoder.

These are correctness demonstrations, not throughput or model-quality benchmarks. Rerunning a notebook creates fresh runtime owners. See the corresponding page in `docs/tutorials/` for ownership, failure, retention, and deployment boundaries.